## Imports and Configuration

In [6]:
# Core
import pandas as pd
import numpy as np

# Dates
from datetime import datetime, timedelta

# Optional but VERY helpful later for holidays
import holidays

# Display tweaks (makes debugging way nicer)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# File path
DATA_FILE = "Cycle_1_TOU_Manual_thru_4_20.xlsx"  # <-- change this

# Toggle for test mode
TEST_MODE = False
TEST_METERS = 5   # number of meters to include in test
TEST_DAYS = 100     # number of days per meter

## Load Data

In [8]:
df = pd.read_excel(DATA_FILE)

print("Shape:", df.shape)
df.head()

Shape: (111733, 12)


,MeterIdentifier,AccountNumber,AccountSubNumber,AccountRate,Multiplier,LocationNumber,MeterPosition,ReadLogDate,ReadValue,ReadDate,UOM,CISCycleIdentifier
0,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.073,2026-03-14 18:15:00,KWH,1
1,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.056,2026-03-14 18:30:00,KWH,1
2,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.050,2026-03-14 18:45:00,KWH,1
3,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.075,2026-03-14 19:00:00,KWH,1
4,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.100,2026-03-14 19:15:00,KWH,1


In [9]:
# SCRATCH CHECKING
print("Unique multipliers:", df["Multiplier"].nunique())
print(df["Multiplier"].unique())
#Unique cycles
print("Unique cycles:", df["CISCycleIdentifier"].nunique())
print(df["CISCycleIdentifier"].unique())

Unique multipliers: 5
[  1.  nan 100. 160.  80.  40.]
Unique cycles: 2
[1 9]


## Basic Cleaning and Typing

In [10]:
print("Before cleaning:", df.shape)

# Convert datetime
df["ReadLogDate"] = pd.to_datetime(df["ReadLogDate"], errors="coerce")
df["ReadDate"] = pd.to_datetime(df["ReadDate"], errors="coerce")

# Numeric safety
df["ReadValue"] = pd.to_numeric(df["ReadValue"], errors="coerce")
df["Multiplier"] = pd.to_numeric(df["Multiplier"], errors="coerce")

# Drop obvious garbage rows
df = df.dropna(subset=["ReadDate", "ReadValue"])

# Sort for sanity
df = df.sort_values(["MeterIdentifier", "ReadDate"]).reset_index(drop=True)

print("After cleaning:", df.shape)

Before cleaning: (111733, 12)
After cleaning: (111733, 12)


## Sanity Check

In [11]:
print("Unique meters:", df["MeterIdentifier"].nunique())
print("Date range:", df["ReadDate"].min(), "→", df["ReadDate"].max())

# Check interval spacing (sample one meter)
sample_meter = df["MeterIdentifier"].iloc[0]
sample = df[df["MeterIdentifier"] == sample_meter].copy()

sample["delta"] = sample["ReadDate"].diff()
sample["delta_minutes"] = sample["delta"] / pd.Timedelta(minutes=1)
print(sample["delta"].value_counts().head())
print(sample["delta_minutes"].value_counts().head())

Unique meters: 30
Date range: 2026-03-06 12:15:00 → 2026-04-20 00:00:00
delta
0 days 00:15:00    3347
0 days 08:45:00       1
1 days 00:15:00       1
Name: count, dtype: int64
delta_minutes
15.0      3347
525.0        1
1455.0       1
Name: count, dtype: int64


# Create Run Subset

In [12]:
if TEST_MODE:
    # Pick a few meters
    meters = df["MeterIdentifier"].dropna().unique()[:TEST_METERS]
    test_df = df[df["MeterIdentifier"].isin(meters)].copy()

    # Limit to first N days per meter
    test_df["date"] = test_df["ReadDate"].dt.date
    
    test_df = (
        test_df.sort_values("ReadDate")
        .groupby("MeterIdentifier")
        .apply(lambda x: x[x["date"] <= (x["date"].min() + timedelta(days=TEST_DAYS))])
        .reset_index(drop=True)
    )

    df_test = test_df.drop(columns=["date"])

    print("Test shape:", df_test.shape)
else:
    df_test = df.copy()

## Base Time Columns

In [13]:
df_test["hour"] = df_test["ReadDate"].dt.hour
df_test["minute"] = df_test["ReadDate"].dt.minute
df_test["date"] = df_test["ReadDate"].dt.date
df_test["month"] = df_test["ReadDate"].dt.month
df_test["weekday"] = df_test["ReadDate"].dt.weekday  # 0=Mon, 6=Sun

# Weekend flag
df_test["is_weekend"] = df_test["weekday"] >= 5

## Flag Meter Reads Missing Multipliers and Fill with 1s

In [14]:
df_test["missing_multiplier"] = df_test["Multiplier"].isna()
df_test["Multiplier"] = df_test["Multiplier"].fillna(1)

## Calc Multiplied Usage

In [15]:
df_test["usage_kwh_calc"] = df_test["ReadValue"] * df_test["Multiplier"]
print(df_test.shape)

(111733, 20)


## Set Up Holidays

In [16]:
# US holidays (you can customize later)
us_holidays = holidays.US()

def get_holiday_set(years):
    holiday_dates = set()
    for y in years:
        for date in holidays.US(years=y).keys():
            holiday_dates.add(date)
    return holiday_dates

years_in_data = df_test["ReadLogDate"].dt.year.unique()
holiday_dates = get_holiday_set(years_in_data)

len(holiday_dates)

12

In [17]:
df_test.head(10)


,MeterIdentifier,AccountNumber,AccountSubNumber,AccountRate,Multiplier,LocationNumber,MeterPosition,ReadLogDate,ReadValue,ReadDate,UOM,CISCycleIdentifier,hour,minute,date,month,weekday,is_weekend,missing_multiplier,usage_kwh_calc
0,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.073,2026-03-14 18:15:00,KWH,1,18,15,2026-03-14,3,5,True,False,0.073
1,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.056,2026-03-14 18:30:00,KWH,1,18,30,2026-03-14,3,5,True,False,0.056
2,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.050,2026-03-14 18:45:00,KWH,1,18,45,2026-03-14,3,5,True,False,0.050
3,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.075,2026-03-14 19:00:00,KWH,1,19,0,2026-03-14,3,5,True,False,0.075
4,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.100,2026-03-14 19:15:00,KWH,1,19,15,2026-03-14,3,5,True,False,0.100
5,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.068,2026-03-14 19:30:00,KWH,1,19,30,2026-03-14,3,5,True,False,0.068
6,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.058,2026-03-14 19:45:00,KWH,1,19,45,2026-03-14,3,5,True,False,0.058
7,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.058,2026-03-14 20:00:00,KWH,1,20,0,2026-03-14,3,5,True,False,0.058
8,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.089,2026-03-14 20:15:00,KWH,1,20,15,2026-03-14,3,5,True,False,0.089
9,142856004,216362.0,1.0,108,1.0,184721,NaN,2026-03-14,0.127,2026-03-14 20:30:00,KWH,1,20,30,2026-03-14,3,5,True,False,0.127


# Core Functions

In [23]:
# ON/OFF-PEAK CLASSIFICATION
def is_on_peak(row):
    dt = row["ReadDate"]
    
    hour = dt.hour
    month = dt.month
    day = dt.day
    weekday = dt.weekday()  # 0=Mon, 6=Sun
    
    # Weekend → OFF PEAK
    if weekday >= 5:
        return False
    
    # Good Friday (2026-04-03) → OFF PEAK
    if dt.date() == datetime(2026, 4, 3).date():
        return False
    
    # Jan–Mar → 6–9 AM
    if month in [1, 2, 3]:
        return 6 <= hour < 9
    
    # Apr 1–15 → 6–9 AM AND 1–6 PM
    if month == 4 and day <= 15:
        return (6 <= hour < 9) or (13 <= hour < 18)

    # Apr 16–30 → 1–6 PM
    if month == 4 and day >= 16:
        return (13 <= hour < 18)

    # May–Sep → 1–6 PM
    if month in [5, 6, 7, 8, 9]:
        return (13 <= hour < 18)
        return False

In [24]:
# Test if Good Friday (Friday, normally has on-peak) is wholly off-peak

df_test["on_peak"] = df_test.apply(is_on_peak, axis=1)

df_test[
    df_test["ReadDate"].dt.date == datetime(2026, 4, 3).date()
]["on_peak"].value_counts()


on_peak
False    2880
Name: count, dtype: int64

## Generate Summary

In [25]:
summary = df_test.groupby(["MeterIdentifier", "on_peak", "Multiplier", "missing_multiplier"]).agg(usage_kwh_calc_sum=("usage_kwh_calc", "sum"), usage_kwh_raw_sum=("ReadValue", "sum")).reset_index()
print(summary)

    MeterIdentifier  on_peak  Multiplier  missing_multiplier  usage_kwh_calc_sum  usage_kwh_raw_sum
0         142856004    False         1.0               False         2283.844000        2283.844000
1         142856004     True         1.0               False          100.776000         100.776000
2         142856005    False         1.0               False         2082.615000        2082.615000
3         142856005     True         1.0               False          221.416000         221.416000
4         143072967    False         1.0               False            8.847000           8.847000
..              ...      ...         ...                 ...                 ...                ...
58        223301158     True        40.0               False          656.840000          16.421000
59        223360865    False         1.0               False         4449.549000        4449.549000
60        223360865     True         1.0               False          771.734000         771.734000


In [33]:
# --- TRANSFORM LONG → WIDE (ONE ROW PER METER) ---

# 1. Define all the metadata fields you want to keep
# This includes the cycle ID, account info, and multiplier flags
meta_fields = [
    "AccountNumber",
    "AccountSubNumber",
    "AccountRate",
    "CISCycleIdentifier",  # The one you requested
    "Multiplier",
    "missing_multiplier"
]

# 2. Extract these fields from df_test (one row per meter)
# We use 'first' because these values should be identical across all rows for a meter
meter_meta = df_test.groupby("MeterIdentifier").agg({
    col: "first" for col in meta_fields if col in df_test.columns
}).reset_index()

# 3. Pivot the usage data so on-peak/off-peak are side-by-side
pivot_df = summary.pivot_table(
    index="MeterIdentifier",
    columns="on_peak",
    values=["usage_kwh_calc_sum", "usage_kwh_raw_sum"],
    aggfunc="sum"
)

# 4. Flatten columns (e.g., usage_kwh_calc_sum_on_peak)
pivot_df.columns = [
    f"{metric}_{'on_peak' if peak else 'off_peak'}"
    for metric, peak in pivot_df.columns
]

# 5. Merge the metadata and the usage data together
final_df = pivot_df.reset_index().merge(meter_meta, on="MeterIdentifier", how="left")

# 6. Fill missing usage with 0 and organize columns for the final CSV
final_df = final_df.fillna(0)

# Organize columns: ID first, then Meta, then Usage totals
usage_cols = [
    "usage_kwh_calc_sum_off_peak", "usage_kwh_calc_sum_on_peak",
    "usage_kwh_raw_sum_off_peak", "usage_kwh_raw_sum_on_peak"
]
column_order = ["MeterIdentifier"] + [f for f in meta_fields if f in final_df.columns] + usage_cols
final_df = final_df[column_order]

# Save the truly consolidated file
final_df.to_csv("summary_consolidated.csv", index=False)

print(f"Consolidated summary created with {len(final_df)} meters.")
final_df.head()


Consolidated summary created with 30 meters.


,MeterIdentifier,AccountNumber,AccountSubNumber,AccountRate,CISCycleIdentifier,Multiplier,missing_multiplier,usage_kwh_calc_sum_off_peak,usage_kwh_calc_sum_on_peak,usage_kwh_raw_sum_off_peak,usage_kwh_raw_sum_on_peak
0,142856004,216362.0,1.0,108,1,1.0,False,2283.8440,100.776000,2283.8440,100.776000
1,142856005,212624.0,1.0,108,1,1.0,False,2082.6150,221.416000,2082.6150,221.416000
2,143072967,200020.0,114.0,1TW,9,1.0,False,8.8470,1.701000,8.8470,1.701000
3,143072969,10993.0,1.0,103,1,1.0,False,1108.4795,79.357167,1108.4795,79.357167
4,143072970,11413.0,1.0,108,1,1.0,False,2346.2880,178.136000,2346.2880,178.136000


In [34]:
final_df.to_csv("summary_consolidated.csv", index=False)